In [1]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from pathlib import Path
from collections import defaultdict
from scipy import stats
from sklearn.model_selection import KFold

from src.utils.config import settings
from src.utils.logger import get_logger

log = get_logger("cv_ab")
sns.set_theme(style="whitegrid")

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu')

PROC = '../../data/processed/'
FEAT = '../../data/features/'

print(f"✅ Imports ready")
print(f"   Device  : {device}")

✅ Imports ready
   Device  : cpu


In [2]:
import ast

ratings = pd.read_csv(
    PROC + 'ratings_cleaned.csv')
movies  = pd.read_csv(
    PROC + 'movies_master.csv',
    low_memory=False)
movies  = movies[
    movies['movieId'].notna()].copy()
movies['movieId'] = \
    movies['movieId'].astype(int)

def safe_parse(val):
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else []
    except:
        return []

movies['genres_list'] = \
    movies['genres_list'].apply(safe_parse)

# Load vocab + sequences
vocab = joblib.load(
    '../../models/checkpoints/'
    'item_vocabulary.joblib')
user_sequences = joblib.load(
    '../../models/checkpoints/'
    'user_sequences.joblib')

PAD_TOKEN   = vocab['PAD']
OFFSET      = 3
vocab_size  = vocab['vocab_size']
movie2token = vocab['movie2token']
token2movie = {
    int(k): v
    for k, v in vocab['token2movie'].items()
}

# Maps
genre_map = dict(zip(
    movies['movieId'],
    movies['genres_list']))
pop_map = ratings.groupby('movieId')[
    'rating'].count().to_dict()

all_mids = list(
    set(ratings['movieId'].unique()))

print(f"✅ Data loaded")
print(f"   Ratings : {len(ratings):,}")
print(f"   Users   : "
      f"{ratings['userId'].nunique():,}")

✅ Data loaded
   Ratings : 100,004
   Users   : 671


In [3]:
#  Redefine HSTU + Rec Functions
class HSTULayer(nn.Module):
    def __init__(self, embed_dim,
                 n_heads, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_heads   = n_heads
        self.head_dim  = embed_dim // n_heads
        self.q_proj    = nn.Linear(
            embed_dim, embed_dim, bias=False)
        self.k_proj    = nn.Linear(
            embed_dim, embed_dim, bias=False)
        self.v_proj    = nn.Linear(
            embed_dim, embed_dim, bias=False)
        self.out_proj  = nn.Linear(
            embed_dim, embed_dim, bias=False)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff    = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim*4, embed_dim),
            nn.Dropout(dropout))
        self.dropout = nn.Dropout(dropout)
        self.scale   = self.head_dim ** -0.5

    def forward(self, x, src_mask=None):
        B, L, D  = x.shape
        residual = x
        x        = self.norm1(x)
        q = self.q_proj(x).view(
            B,L,self.n_heads,
            self.head_dim).transpose(1,2)
        k = self.k_proj(x).view(
            B,L,self.n_heads,
            self.head_dim).transpose(1,2)
        v = self.v_proj(x).view(
            B,L,self.n_heads,
            self.head_dim).transpose(1,2)
        attn = torch.matmul(
            q, k.transpose(-2,-1)) * self.scale
        causal = torch.tril(torch.ones(
            L, L, device=x.device)
        ).unsqueeze(0).unsqueeze(0)
        attn = F.relu(attn)
        s    = attn.sum(
            dim=-1, keepdim=True
        ).clamp(min=1e-6)
        attn = attn / s
        attn = self.dropout(attn)
        out  = torch.matmul(attn, v)
        out  = out.transpose(1,2)\
            .contiguous().view(B,L,D)
        x    = residual + self.out_proj(out)
        x    = x + self.ff(self.norm2(x))
        return x


class HSTURanker(nn.Module):
    def __init__(self, vocab_size,
                 embed_dim=128, n_heads=4,
                 n_layers=3, max_seq_len=50,
                 dropout=0.1, sl_dropout=0.3):
        super().__init__()
        self.item_emb = nn.Embedding(
            vocab_size, embed_dim,
            padding_idx=PAD_TOKEN)
        self.pos_emb  = nn.Embedding(
            max_seq_len, embed_dim)
        self.hstu_layers = nn.ModuleList([
            HSTULayer(embed_dim, n_heads,
                      dropout)
            for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(
            embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.rating_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim//2),
            nn.ReLU(),
            nn.Linear(embed_dim//2, 1),
            nn.Sigmoid())
        self.completion_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim//2),
            nn.ReLU(),
            nn.Linear(embed_dim//2, 1),
            nn.Sigmoid())

    def encode_user(self, history):
        B, L = history.shape
        pos  = torch.arange(
            L, device=history.device
        ).unsqueeze(0).expand(B,-1)
        x    = self.item_emb(history) + \
               self.pos_emb(pos)
        x    = self.dropout(x)
        for layer in self.hstu_layers:
            x = layer(x)
        x = self.final_norm(x)
        lengths = (history != PAD_TOKEN)\
            .sum(dim=1) - 1
        lengths = lengths.clamp(min=0)
        return x[torch.arange(B), lengths]

    def predict(self, history, top_k=10):
        with torch.no_grad():
            u   = self.encode_user(history)
            emb = self.item_emb.weight
            sc  = torch.matmul(u, emb.T)
            sc[:, :OFFSET] = float('-inf')
            return torch.topk(sc, top_k, dim=-1)


# Load HSTU
hstu = HSTURanker(vocab_size).to(device)
hstu_path = '../../models/checkpoints/'\
            'hstu_best.pt'
if os.path.exists(hstu_path):
    hstu.load_state_dict(torch.load(
        hstu_path, map_location=device,
        weights_only=True))
    hstu.eval()
    print("✅ HSTU loaded")


def get_hstu_recs(uid, train_set,
                   k=10) -> list:
    try:
        seq = user_sequences.get(uid, [])
        if not seq: return []
        pad_l = 50 - len(seq)
        hist  = torch.LongTensor(
            [[PAD_TOKEN]*pad_l + seq[-50:]]
        ).to(device)
        toks, _ = hstu.predict(
            hist, k + len(seq))
        rated = set(seq)
        recs  = []
        for t in toks[0].cpu().numpy():
            mid = token2movie.get(int(t))
            if mid and int(t) not in rated \
                    and mid not in train_set:
                recs.append(mid)
            if len(recs) >= k:
                break
        return recs or []
    except:
        return []


def get_pop_recs(uid, train_set,
                  k=10) -> list:
    pop_top = [m for m, _ in sorted(
        pop_map.items(),
        key=lambda x: x[1],
        reverse=True)[:200]]
    return [m for m in pop_top
            if m not in train_set][:k]


def get_random_recs(uid, train_set,
                     k=10) -> list:
    cands = [m for m in all_mids
             if m not in train_set]
    if not cands: return []
    return list(np.random.choice(
        cands,
        size=min(k, len(cands)),
        replace=False))


print("✅ Recommendation functions ready")

✅ HSTU loaded
✅ Recommendation functions ready


In [4]:
# Metric Functions
def ndcg_at_k(recs, relevant, k=10):
    dcg  = sum(
        1.0/np.log2(i+2)
        for i, m in enumerate(recs[:k])
        if m in relevant)
    idcg = sum(
        1.0/np.log2(i+2)
        for i in range(
            min(len(relevant), k)))
    return dcg/idcg if idcg > 0 else 0.0

def precision_at_k(recs, relevant, k=10):
    hits = sum(1 for m in recs[:k]
               if m in relevant)
    return hits/k if k > 0 else 0.0

def recall_at_k(recs, relevant, k=10):
    if not relevant: return 0.0
    hits = sum(1 for m in recs[:k]
               if m in relevant)
    return hits/len(relevant)

print("✅ Metrics defined")

✅ Metrics defined


In [5]:
# 5-Fold Cross-Validation
print("5-FOLD CROSS-VALIDATION")
print("=" * 55)
print("""
Why 5-fold CV for recommendation:
  Single split → results may be lucky
  5-fold → test on 5 different subsets
  Report mean ± std → statistically robust
  Low std → stable model ✅
  High std → unstable model ⚠️
""")

K_FOLDS = 5

# Sort by timestamp for temporal folds
ratings_sorted = ratings.sort_values(
    'timestamp').reset_index(drop=True)

fold_size = len(ratings_sorted) // K_FOLDS

cv_results = defaultdict(list)

print(f"Running {K_FOLDS}-fold CV...")
print(f"{'Fold':<6} {'HSTU':<12} "
      f"{'Popularity':<14} {'Random':<10}")
print("─" * 45)

for fold in range(K_FOLDS):
    # Temporal fold boundaries
    val_start = fold * fold_size
    val_end   = val_start + fold_size \
        if fold < K_FOLDS - 1 \
        else len(ratings_sorted)

    val_df   = ratings_sorted.iloc[
        val_start:val_end].copy()
    train_df = pd.concat([
        ratings_sorted.iloc[:val_start],
        ratings_sorted.iloc[val_end:]
    ]).copy()

    # Build train set per user
    train_sets = {}
    for uid, grp in train_df.groupby(
            'userId'):
        train_sets[uid] = set(
            grp['movieId'].values)

    # Evaluate on val users
    fold_ndcg = defaultdict(list)

    val_users = val_df['userId'].unique()
    sample    = np.random.choice(
        val_users,
        size=min(50, len(val_users)),
        replace=False)

    for uid in sample:
        train_set = train_sets.get(uid, set())
        rel = set(val_df[
            (val_df['userId'] == uid) &
            (val_df['rating'] >= 3.5)
        ]['movieId'].values)

        if not rel: continue

        for name, fn in [
            ('hstu',  get_hstu_recs),
            ('pop',   get_pop_recs),
            ('rand',  get_random_recs),
        ]:
            recs = fn(int(uid),
                      train_set, 10)
            if recs:
                fold_ndcg[name].append(
                    ndcg_at_k(recs, rel, 10))

    # Store fold results
    for name in ['hstu', 'pop', 'rand']:
        val = np.mean(fold_ndcg[name]) \
            if fold_ndcg[name] else 0.0
        cv_results[name].append(
            round(val, 4))

    print(
        f"{fold+1:<6} "
        f"{cv_results['hstu'][-1]:<12.4f} "
        f"{cv_results['pop'][-1]:<14.4f} "
        f"{cv_results['rand'][-1]:<10.4f}")

print(f"\n{'─'*45}")
print(f"{'Mean':<6} "
      f"{np.mean(cv_results['hstu']):<12.4f} "
      f"{np.mean(cv_results['pop']):<14.4f} "
      f"{np.mean(cv_results['rand']):<10.4f}")
print(f"{'Std':<6} "
      f"{np.std(cv_results['hstu']):<12.4f} "
      f"{np.std(cv_results['pop']):<14.4f} "
      f"{np.std(cv_results['rand']):<10.4f}")

# Summary DataFrame
cv_summary = pd.DataFrame({
    'model': ['HSTU', 'Popularity', 'Random'],
    'mean_ndcg': [
        round(np.mean(cv_results['hstu']), 4),
        round(np.mean(cv_results['pop']),  4),
        round(np.mean(cv_results['rand']), 4),
    ],
    'std_ndcg': [
        round(np.std(cv_results['hstu']), 4),
        round(np.std(cv_results['pop']),  4),
        round(np.std(cv_results['rand']), 4),
    ],
    'cv_score': [
        round(np.mean(cv_results['hstu']) /
              max(np.std(cv_results['hstu']),
                  1e-6), 2),
        round(np.mean(cv_results['pop'])  /
              max(np.std(cv_results['pop']),
                  1e-6), 2),
        round(np.mean(cv_results['rand']) /
              max(np.std(cv_results['rand']),
                  1e-6), 2),
    ]
})

print(f"\nCROSS-VALIDATION SUMMARY")
print(cv_summary.to_string(index=False))
print(f"""
cv_score = mean/std (higher = more stable)
  High cv_score → consistent across folds
  Low cv_score  → high variance → unstable
""")

cv_summary.to_csv(
    PROC + 'cv_results.csv', index=False)
print("✅ CV results saved")

5-FOLD CROSS-VALIDATION

Why 5-fold CV for recommendation:
  Single split → results may be lucky
  5-fold → test on 5 different subsets
  Report mean ± std → statistically robust
  Low std → stable model ✅
  High std → unstable model ⚠️

Running 5-fold CV...
Fold   HSTU         Popularity     Random    
─────────────────────────────────────────────
1      0.0229       0.4078         0.0105    
2      0.0204       0.2963         0.0236    
3      0.0448       0.4237         0.0123    
4      0.0044       0.2875         0.0112    
5      0.0188       0.3588         0.0051    

─────────────────────────────────────────────
Mean   0.0223       0.3548         0.0125    
Std    0.0130       0.0557         0.0061    

CROSS-VALIDATION SUMMARY
     model  mean_ndcg  std_ndcg  cv_score
      HSTU     0.0223    0.0130      1.71
Popularity     0.3548    0.0557      6.37
    Random     0.0125    0.0061      2.07

cv_score = mean/std (higher = more stable)
  High cv_score → consistent across folds


In [6]:
# Statistical Significance Testing
print("STATISTICAL SIGNIFICANCE TESTING")
print("=" * 55)
print("t-test: is HSTU significantly better?")
print("p < 0.05 = statistically significant\n")

from scipy import stats

# Paired t-test: HSTU vs Popularity
# across folds
hstu_folds = cv_results['hstu']
pop_folds  = cv_results['pop']
rand_folds = cv_results['rand']

def run_ttest(name_a, vals_a,
              name_b, vals_b):
    """Paired t-test between two models"""
    if len(vals_a) < 2 or \
       len(vals_b) < 2:
        return None

    t_stat, p_val = stats.ttest_rel(
        vals_a, vals_b)

    sig = "✅ significant" \
        if p_val < 0.05 \
        else "⚠️  not significant"
    direction = "better" \
        if np.mean(vals_a) > \
           np.mean(vals_b) \
        else "worse"

    print(f"{name_a} vs {name_b}:")
    print(f"  t-statistic : {t_stat:.4f}")
    print(f"  p-value     : {p_val:.4f}")
    print(f"  Result      : {sig}")
    print(f"  Direction   : "
          f"{name_a} is {direction} "
          f"than {name_b}")
    print()

    return {
        "comparison": f"{name_a}_vs_{name_b}",
        "t_stat":     round(t_stat, 4),
        "p_val":      round(p_val, 4),
        "significant":p_val < 0.05,
        "direction":  direction,
    }


sig_results = []

r1 = run_ttest(
    "HSTU", hstu_folds,
    "Random", rand_folds)
if r1: sig_results.append(r1)

r2 = run_ttest(
    "HSTU", hstu_folds,
    "Popularity", pop_folds)
if r2: sig_results.append(r2)

r3 = run_ttest(
    "Popularity", pop_folds,
    "Random", rand_folds)
if r3: sig_results.append(r3)

print(f"""
NOTE: With only 5 folds the t-test has
low statistical power. p > 0.05 does not
mean the difference is not real — it means
we need more data to confirm significance.

Standard practice: report mean ± std
and note p-values as indicative.
Production A/B testing (below) provides
stronger statistical evidence.
""")

sig_df = pd.DataFrame(sig_results)
sig_df.to_csv(
    PROC + 'significance_tests.csv',
    index=False)
print("✅ Significance tests saved")

STATISTICAL SIGNIFICANCE TESTING
t-test: is HSTU significantly better?
p < 0.05 = statistically significant

HSTU vs Random:
  t-statistic : 1.3873
  p-value     : 0.2376
  Result      : ⚠️  not significant
  Direction   : HSTU is better than Random

HSTU vs Popularity:
  t-statistic : -14.4463
  p-value     : 0.0001
  Result      : ✅ significant
  Direction   : HSTU is worse than Popularity

Popularity vs Random:
  t-statistic : 11.6955
  p-value     : 0.0003
  Result      : ✅ significant
  Direction   : Popularity is better than Random


NOTE: With only 5 folds the t-test has
low statistical power. p > 0.05 does not
mean the difference is not real — it means
we need more data to confirm significance.

Standard practice: report mean ± std
and note p-values as indicative.
Production A/B testing (below) provides
stronger statistical evidence.

✅ Significance tests saved


In [7]:
# A/B Test Scaffolding
print("A/B TEST SCAFFOLDING")
print("=" * 55)
print("""
Production A/B testing:
  Control (A): current production model
  Treatment (B): new model being tested
  Traffic split: 90% A, 10% B
  Monitor: NDCG, CTR, watch time, retention
  Decision: promote B if metrics improve
            with statistical significance
""")

import hashlib
from datetime import datetime, timedelta

class ABTestScaffold:
    """
    A/B test scaffolding for recommendation.

    Implements:
    1. User assignment to variants
       → deterministic (same user always A or B)
       → hash-based → no storage needed
    2. Per-variant metric logging
    3. Statistical test on results
    4. Decision framework

    Production version:
    → FastAPI endpoints /recommend/A /recommend/B
    → Nginx traffic splitting
    → Grafana dashboard per variant
    → Automated promotion if p < 0.05
    """

    def __init__(self,
                 experiment_name: str,
                 control_pct:     float = 0.9,
                 treatment_pct:   float = 0.1):
        self.name         = experiment_name
        self.control_pct  = control_pct
        self.treatment_pct= treatment_pct
        self.metrics_a    = defaultdict(list)
        self.metrics_b    = defaultdict(list)
        self.start_time   = datetime.now()

    def assign_variant(self,
                        user_id: int) -> str:
        """
        Assign user to A or B deterministically.
        Same user always gets same variant.
        Uses hash → no database needed.
        """
        h = int(hashlib.md5(
            f"{self.name}_{user_id}"
            .encode()
        ).hexdigest(), 16)

        if (h % 100) < \
                (self.control_pct * 100):
            return 'A'
        return 'B'

    def log_metric(self,
                    user_id: int,
                    metric: str,
                    value: float):
        """Log metric for user's variant"""
        variant = self.assign_variant(user_id)
        if variant == 'A':
            self.metrics_a[metric].append(value)
        else:
            self.metrics_b[metric].append(value)

    def get_results(self,
                     metric: str) -> dict:
        """Get A/B comparison for metric"""
        a_vals = self.metrics_a.get(metric, [])
        b_vals = self.metrics_b.get(metric, [])

        if not a_vals or not b_vals:
            return {
                "metric": metric,
                "status": "insufficient data"
            }

        a_mean = np.mean(a_vals)
        b_mean = np.mean(b_vals)
        lift   = (b_mean - a_mean) / \
                  max(a_mean, 1e-10) * 100

        # Welch's t-test
        # (unequal sample sizes allowed)
        if len(a_vals) >= 2 and \
           len(b_vals) >= 2:
            t, p = stats.ttest_ind(
                a_vals, b_vals,
                equal_var=False)
        else:
            p = 1.0

        decision = "promote B" \
            if (lift > 0 and p < 0.05) \
            else "keep A" \
            if lift < 0 \
            else "collect more data"

        return {
            "metric":        metric,
            "A_mean":        round(a_mean, 4),
            "B_mean":        round(b_mean, 4),
            "lift_pct":      round(lift, 2),
            "p_value":       round(p, 4),
            "significant":   p < 0.05,
            "n_A":           len(a_vals),
            "n_B":           len(b_vals),
            "decision":      decision,
        }

    def should_promote(self,
                        metric: str) -> bool:
        """True if B should replace A"""
        r = self.get_results(metric)
        return r.get('decision') == "promote B"

    def summary(self) -> str:
        duration = datetime.now() - \
                   self.start_time
        a_users  = sum(
            1 for uid in range(1000)
            if self.assign_variant(uid) == 'A')
        b_users  = 1000 - a_users
        return (
            f"Experiment: {self.name}\n"
            f"Duration  : {duration}\n"
            f"Split     : "
            f"{a_users/10:.0f}% A / "
            f"{b_users/10:.0f}% B\n"
            f"Metrics   : "
            f"{list(self.metrics_a.keys())}"
        )


print("✅ A/B test scaffold defined")

A/B TEST SCAFFOLDING

Production A/B testing:
  Control (A): current production model
  Treatment (B): new model being tested
  Traffic split: 90% A, 10% B
  Monitor: NDCG, CTR, watch time, retention
  Decision: promote B if metrics improve
            with statistical significance

✅ A/B test scaffold defined


In [8]:
# Run A/B Simulation
print("RUNNING A/B TEST SIMULATION")
print("=" * 55)
print("A = Popularity (current production)")
print("B = HSTU (new model being tested)")
print("Traffic split: 90% A / 10% B\n")

# Create experiment
ab_test = ABTestScaffold(
    experiment_name = "hstu_vs_popularity",
    control_pct     = 0.9,
    treatment_pct   = 0.1,
)

# Simulate users interacting with the system
test_users = ratings['userId'].unique()
np.random.seed(42)
sim_users  = np.random.choice(
    test_users,
    size=min(500, len(test_users)),
    replace=False)

# Build user train sets
train_df_ab = ratings.sort_values(
    'timestamp').iloc[
    :int(len(ratings)*0.8)].copy()

test_df_ab = ratings.sort_values(
    'timestamp').iloc[
    int(len(ratings)*0.8):].copy()

train_sets_ab = {}
for uid, grp in train_df_ab.groupby(
        'userId'):
    train_sets_ab[uid] = set(
        grp['movieId'].values)

# Log metrics per variant
n_a = n_b = 0
for uid in sim_users:
    variant = ab_test.assign_variant(uid)
    train_s = train_sets_ab.get(uid, set())

    rel = set(test_df_ab[
        (test_df_ab['userId'] == uid) &
        (test_df_ab['rating'] >= 3.5)
    ]['movieId'].values)

    if not rel: continue

    # Get recs from assigned variant
    if variant == 'A':
        recs = get_pop_recs(
            int(uid), train_s, 10)
        n_a += 1
    else:
        recs = get_hstu_recs(
            int(uid), train_s, 10)
        n_b += 1

    if not recs: continue

    ndcg = ndcg_at_k(recs, rel, 10)
    prec = precision_at_k(recs, rel, 10)

    ab_test.log_metric(uid, 'ndcg@10', ndcg)
    ab_test.log_metric(uid, 'precision@10',
                       prec)

print(f"Simulated {n_a + n_b} user sessions")
print(f"  Variant A (Popularity) : {n_a}")
print(f"  Variant B (HSTU)       : {n_b}")

# Results
print(f"\nA/B TEST RESULTS")
print("=" * 55)
for metric in ['ndcg@10', 'precision@10']:
    r = ab_test.get_results(metric)
    print(f"\nMetric: {metric}")
    print(f"  A (Popularity) : "
          f"{r.get('A_mean', 0):.4f} "
          f"(n={r.get('n_A', 0)})")
    print(f"  B (HSTU)       : "
          f"{r.get('B_mean', 0):.4f} "
          f"(n={r.get('n_B', 0)})")
    print(f"  Lift           : "
          f"{r.get('lift_pct', 0):+.2f}%")
    print(f"  p-value        : "
          f"{r.get('p_value', 1):.4f}")
    print(f"  Decision       : "
          f"{r.get('decision', 'unknown')}")

print(f"\n{ab_test.summary()}")

RUNNING A/B TEST SIMULATION
A = Popularity (current production)
B = HSTU (new model being tested)
Traffic split: 90% A / 10% B

Simulated 116 user sessions
  Variant A (Popularity) : 98
  Variant B (HSTU)       : 18

A/B TEST RESULTS

Metric: ndcg@10
  A (Popularity) : 0.3990 (n=98)
  B (HSTU)       : 0.0000 (n=18)
  Lift           : -100.00%
  p-value        : 0.0000
  Decision       : keep A

Metric: precision@10
  A (Popularity) : 0.3735 (n=98)
  B (HSTU)       : 0.0000 (n=18)
  Lift           : -100.00%
  p-value        : 0.0000
  Decision       : keep A

Experiment: hstu_vs_popularity
Duration  : 0:00:00.612218
Split     : 90% A / 10% B
Metrics   : ['ndcg@10', 'precision@10']


In [ ]:
# 